In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import random

class MNISTSequenceDataset(Dataset):
    def __init__(self, mnist_dataset, min_seq_len=3, max_seq_len=5):
        self.mnist = mnist_dataset
        self.min_seq_len = min_seq_len
        self.max_seq_len = max_seq_len

    def __len__(self):
        return 5000

    def __getitem__(self, idx):
        seq_len = random.randint(self.min_seq_len, self.max_seq_len)
        indices = [random.randint(0, len(self.mnist) - 1) for _ in range(seq_len)]
        
        images = [self.mnist[i][0] for i in indices]
        labels = [self.mnist[i][1] for i in indices]
        
        concat_image = torch.cat(images, dim=2)
        
        return concat_image, torch.tensor(labels, dtype=torch.long)

def collate_fn(batch):
    images, targets = zip(*batch)
    
    max_width = max(img.shape[2] for img in images)
    padded_images = []
    input_lengths = []
    
    for img in images:
        p_width = max_width - img.shape[2]
        padded_img = nn.functional.pad(img, (0, p_width, 0, 0))
        padded_images.append(padded_img)
        input_lengths.append(padded_img.shape[2] // 4)
        
    padded_images = torch.stack(padded_images)
    
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    concat_targets = torch.cat(targets)
    
    return padded_images, concat_targets, torch.tensor(input_lengths, dtype=torch.long), target_lengths

class CRNN(nn.Module):
    def __init__(self, num_classes=11):
        super(CRNN, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.rnn = nn.GRU(64 * 7, 128, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(128 * 2, num_classes)

    def forward(self, x):
        features = self.cnn(x)
        b, c, h, w = features.size()
        features = features.permute(0, 3, 1, 2).contiguous()
        features = features.view(b, w, c * h)
        
        rnn_out, _ = self.rnn(features)
        output = self.fc(rnn_out)
        
        return output.permute(1, 0, 2)

def main():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    
    mnist_raw = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    dataset = MNISTSequenceDataset(mnist_raw)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model = CRNN(num_classes=11).to(device)
    ctc_loss = nn.CTCLoss(blank=10, zero_infinity=True)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    num_epochs = 5
    model.train()
    
    for epoch in range(num_epochs):
        running_loss = 0.0
        for inputs, targets, input_lengths, target_lengths in dataloader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            optimizer.zero_grad()
            
            outputs = model(inputs)
            log_probs = nn.functional.log_softmax(outputs, dim=2)
            
            loss = ctc_loss(log_probs, targets, input_lengths, target_lengths)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(dataloader):.4f}")

if __name__ == "__main__":
    main()

100%|██████████| 9.91M/9.91M [00:05<00:00, 1.71MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 147kB/s]
100%|██████████| 1.65M/1.65M [00:02<00:00, 824kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.35MB/s]


Epoch [1/5], Loss: 2.3406
Epoch [2/5], Loss: 0.3168
Epoch [3/5], Loss: 0.1180
Epoch [4/5], Loss: 0.0831
Epoch [5/5], Loss: 0.0631
